### Criando tabelas e inserindo dados no sqlite3
- https://docs.python.org/3/library/sqlite3.html

In [1]:
# Importando o sqlite
import sqlite3
con = sqlite3.connect('../data/banco.db')

In [2]:
# Criando uma conexão
cur = con.cursor()

In [3]:
# Criando o cursor
#cur.execute("CREATE TABLE movie(title, year, score)")

In [4]:
# Selecionando todos os dados dessa tabela criada
cur.execute("SELECT * FROM movie").fetchall()

[('Monty Python and the Holy Grail', 1975, 8.2),
 ('And Now for Something Completely Different', 1971, 7.5)]

In [5]:
# Salvando as mudanças
#con.commit()

In [6]:
# Fechando a conexão
#con.close()

### Adicionando um DataFrame como tabela na nossa base
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html

In [7]:
# Importando o pandas
import pandas as pd

In [8]:
dados = {
    'X': [1,2,3,4,5],
    'Y': [10,9,8,7,6],
    'Z': ['a','b','c','d','e']
}

dados = pd.DataFrame(dados)

In [39]:
# Visualizando esse DataFrame
dados

,X,Y,Z
0,1,10,a
1,2,9,b
2,3,8,c
3,4,7,d
4,5,6,e


In [40]:
# Abrindo novamente a conexão
con = sqlite3.connect('../data/exemplo.db')

In [41]:
# Adicionando essa base como uma tabela
dados.to_sql('dados', con)

5

In [42]:
# Criando o cursor
cur = con.cursor()

In [45]:
# Selecionando todos os dados dessa tabela criada
cur.execute('SELECT * FROM dados').fetchall()

[(0, 1, 10, 'a'),
 (1, 2, 9, 'b'),
 (2, 3, 8, 'c'),
 (3, 4, 7, 'd'),
 (4, 5, 6, 'e')]

In [ ]:
# Verificando o nome das colunas
cur.description

(('index', None, None, None, None, None, None),
 ('X', None, None, None, None, None, None),
 ('Y', None, None, None, None, None, None),
 ('Z', None, None, None, None, None, None))

In [49]:
# Adicionando como tabela mas sem o index
dados.to_sql('dados', con, index=False, if_exists='replace')

5

In [50]:
cur.execute('SELECT * FROM dados').fetchall()

[(1, 10, 'a'), (2, 9, 'b'), (3, 8, 'c'), (4, 7, 'd'), (5, 6, 'e')]

In [51]:
cur.description

(('X', None, None, None, None, None, None),
 ('Y', None, None, None, None, None, None),
 ('Z', None, None, None, None, None, None))

**Ao tentar criar uma nova tabela com o mesmo nome, teremos o erro mostrado abaixo:**<br>
`ValueError: Table 'dados' already exists.`

### DROP TABLE
- O `DROP TABLE` permite apagarmos qualquer tabela do nosso banco de dados

In [ ]:
# Executando o DROP dessa tabela
# cur.execute('DROP TABLE dados')

### Inserindo novos valores

In [52]:
dados2 = {
    'X': [6,7,8],
    'Y': [5,4,3],
    'Z': ['f','l','h']
}

dados2 = pd.DataFrame(dados2)

dados2.head()

,X,Y,Z
0,6,5,f
1,7,4,l
2,8,3,h


**O to_sql aceita um parâmetro para caso já exista a tabela**
- O `if_exists` permite fazer um adição dos dados caso a tabela já exista
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html

In [54]:
# Usando o append da base
dados2.to_sql('dados', con, index=False, if_exists='append')

3

In [55]:
# Fazendo o SELECT
cur.execute('SELECT * FROM dados').fetchall()

[(1, 10, 'a'),
 (2, 9, 'b'),
 (3, 8, 'c'),
 (4, 7, 'd'),
 (5, 6, 'e'),
 (6, 5, 'f'),
 (7, 4, 'l'),
 (8, 3, 'h')]

**Também podemos adicionar valores utilizando o `INSERT`**

In [56]:
# Podemos adicionar diretamente os valores
# Adicionando o valor (9,2,'j')
cur.execute('INSERT INTO  dados values (9,2,"j")')

In [57]:
# Fazendo o SELECT
cur.execute('SELECT * FROM dados').fetchall()

[(1, 10, 'a'),
 (2, 9, 'b'),
 (3, 8, 'c'),
 (4, 7, 'd'),
 (5, 6, 'e'),
 (6, 5, 'f'),
 (7, 4, 'l'),
 (8, 3, 'h'),
 (9, 2, 'j')]

**Além disso, podemos utilizar placeholders para adicionar os valores**
- https://docs.python.org/3/library/sqlite3.html#using-placeholders-to-bind-values-in-sql-queries

In [63]:
# Utilizando placeholders para adicionar os valores (10,1,'j')
cur.execute('INSERT INTO  dados values (?,?,?)', (10,1,"j"))

In [64]:
cur.execute('SELECT * FROM dados').fetchall()

[(1, 10, 'a'),
 (2, 9, 'b'),
 (3, 8, 'c'),
 (4, 7, 'd'),
 (5, 6, 'e'),
 (6, 5, 'f'),
 (7, 4, 'l'),
 (8, 3, 'h'),
 (9, 2, 'j'),
 (10, 1, 'j'),
 (10, 1, 'j'),
 (10, 1, 'j')]

In [65]:
cur.execute("""
DELETE FROM dados
WHERE rowid NOT IN (
    SELECT MIN(rowid)
    FROM dados
    GROUP BY X, Y, Z
)
""")

con.commit()

In [66]:
cur.execute("SELECT * FROM dados").fetchall()

[(1, 10, 'a'),
 (2, 9, 'b'),
 (3, 8, 'c'),
 (4, 7, 'd'),
 (5, 6, 'e'),
 (6, 5, 'f'),
 (7, 4, 'l'),
 (8, 3, 'h'),
 (9, 2, 'j'),
 (10, 1, 'j')]

**Ou utilizar o `.execute_many()` para adicionar valores de uma lista**
- https://docs.python.org/3/library/sqlite3.html#sqlite3.Cursor.executemany

In [67]:
adicionar = [
    (11,0,'k'),
    (12,-1,'l'),
    (13,-2,'m')
]

In [68]:
# Adicionando esses valores
cur.executemany('INSERT INTO dados values  (?,?,?)', adicionar)

In [69]:
cur.execute("SELECT * FROM dados").fetchall()

[(1, 10, 'a'),
 (2, 9, 'b'),
 (3, 8, 'c'),
 (4, 7, 'd'),
 (5, 6, 'e'),
 (6, 5, 'f'),
 (7, 4, 'l'),
 (8, 3, 'h'),
 (9, 2, 'j'),
 (10, 1, 'j'),
 (11, 0, 'k'),
 (12, -1, 'l'),
 (13, -2, 'm')]

### UPDATE e DELETE
- Com o `UPDATE` podemos atualizar qualquer registro da nossa base de dados
- Já o `DELETE` permite eliminar qualquer registro (ou todos os registros)
- <font color='red'>**CUIDADO! NUNCA ESQUEÇAM O WHERE ANTES DE UTILIZAR ESSE COMANDO**</font>

In [28]:
# Visualizando a nossa tabela

In [29]:
# Atualizando a linha 7 para a letra g ao invés da letra l

In [30]:
# Retornando todos os valores

**Vamos executar apagar a nossa tabela dados e executar novamente os comandos acima para voltar com a tabela como estava sem executar o update**

In [31]:
# Apagando a tabela

[Voltar](#voltar)

In [32]:
# Primeiramente vamos apenas filtrar as linhas que queremos atualizar

In [33]:
# E agora vamos atualizar apenas essa linha

In [34]:
# E novamente visualizar toda a tabela

**E para deletar uma linha podemos usar a mesma lógica**

In [35]:
# Deletando as últimas 2 linhas

In [36]:
# Visualizando os dados